# 05. 다운샘플링 실험 — 셀별 실행

스크립트(`05_sampling_*.py`)의 함수를 그대로 부른다. 코드 수정은 `.py` 에서 하고 여기선 실행·확인만.

```
02 ×3 (구간별 피처)  →  select (방식 × 비율)  →  audit (통계)  →  통계 선별  →  LightGBM 빠른 비교(예정)  →  GNN train (추린 조건만)
```

**시간 분할** (월~일 주 단위, 11월 이후 미사용 — 11/6 부터 패턴 외 세탁 0건·정상 급감하는 생성기 꼬리 구간)

| 구간 | 기간 | 쓰임 |
|---|---|---|
| 학습 | 2022-08-08 ~ 10-02 (8주) | 학습 target. 8/1~8/7 은 그래프·이력에만 |
| 검증 | 2022-10-03 ~ 10-16 (2주) | 에폭·임계값 선택 |
| 테스트 | 2022-10-17 ~ 10-30 (2주) | 최종 후보만 한 번 평가 (`--eval-test`) |

**누수 방지** — 노드 피처와 adjacency(이웃 후보)는 각 구간 끝까지의 거래로만 만든다 (원 논문 방식).
학습 target 은 `features_train`(~10/02), 검증은 `features_val`(~10/16), 테스트는 `features_test`(~10/30) 의 그래프·피처를 쓴다.
엣지 피처는 원래 각 거래 직전까지만 보므로 구간과 무관하게 같다.
스크립트가 시작할 때 `features.json` 의 `context_end` 가 구간 끝과 같은지 확인하고 다르면 멈춘다.

실험 조건 = select(방식×비율) × apply(loss|struct) × reweight(none) × pi_eff × seed
- apply=loss : 그래프 유지, loss 에 넣는 정상 target 만 줄임 (지도 샘플링) / struct : adjacency 에서도 제거
- pi_eff : 손실에서 양성 가중치 비중. pos_weight 를 여기서 역산해 r 과 분리 (ipw 와 none 은 같은 목적함수 → none 만)

## 0. 설정

In [ ]:
!pip install --break-system-packages --no-cache-dir -q torch_geometric

In [ ]:
import importlib, json, time, sys
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

import sampling_lib as L
select  = importlib.import_module("05_sampling_select")
audit   = importlib.import_module("05_sampling_audit")
train   = importlib.import_module("05_sampling_train")
collect = importlib.import_module("05_sampling_collect")

THREADS, MEMORY = 8, "12GB"          # cgroup 한도 27GB — DuckDB 한도 + 파이썬 배열이 그 안에 들어야 함
TRAIN, VAL, TEST = L.TRAIN, L.VAL, L.TEST     # 2022-08-08:10-02 / 10-03:10-16 / 10-17:10-30
FEAT = L.FEAT                                 # {"train": "features_train", "val": "features_val", "test": "features_test"}

# 단계가 끝날 때마다 여기만 바꾼다 — 나머지 셀은 이 두 값으로 이름을 조합한다
R    = "10"                 # 정상:양성 비율
BEST = f"random_r{R}"       # 선별 후 최적 select 이름 (예: f"strata_r{R}_day-amount-cross_bank", f"hard_r{R}")
print("ROOT", L.ROOT); print("BASE", L.BASE); print("출력", L.SAMPLING)
print(f"학습 {TRAIN} / 검증 {VAL} / 테스트 {TEST}")
for f in ["transactions.parquet", "patterns.parquet"]:
    print(f"{'OK ' if (L.BASE/f).is_file() else '없음'} {f}")


### 0-1. 구간별 피처 만들기 — `02_build_features.ipynb` 를 3번

`02` 의 설정 셀에서 아래 두 값만 바꿔 각각 전체 실행한다 (코드 수정 없음, 1회당 수십 분~1시간).

| TAG | CONTEXT_END | 출력 폴더 | 필요한 단계 |
|---|---|---|---|
| `"train"` | `"2022-10-02"` | `parquets/HI-Large/features_train/` | select(strata·cssmc)·audit·train — **먼저** |
| `"val"`   | `"2022-10-16"` | `parquets/HI-Large/features_val/`   | train (검증 예측) |
| `"test"`  | `"2022-10-30"` | `parquets/HI-Large/features_test/`  | train `--eval-test` (최종 후보만) |

통계 선별까지는 `features_train` 만 있으면 된다. 기존 `features/`(전체 기간, 누수 있음)는 이 실험에서 쓰지 않는다.

In [ ]:
# 구간별 피처 준비 상태 — context_end 가 구간 끝과 같아야 한다
for split, win in [("train", TRAIN), ("val", VAL), ("test", TEST)]:
    name = FEAT[split]; end = L.parse_window(win)[1]
    try:
        ce = L.features_context_end(name)
        print(f"{'OK ' if ce == end else '불일치'}  {name:15s} context_end={ce}  (구간 끝 {end})")
    except AssertionError:
        print(f"없음  {name:15s} → 02 를 TAG='{split}', CONTEXT_END='{end}' 로 실행")


### 학습 구간 라벨 분포 — 일별 양성 수와 정상 수 (r 범위 정하는 근거)

In [ ]:
con = L.connect(THREADS, MEMORY)
lab = L.load_labels(con, L.parse_window(TRAIN))
daily = lab.groupby("day").agg(pos=("y", "sum"), n=("y", "size"), unmatched=("in_pattern", lambda s: int(((lab.loc[s.index, "y"] == 1) & ~s).sum())))
daily["neg"] = daily.n - daily.pos; daily["neg_per_pos"] = (daily.neg / daily.pos).round(0)
print(f"학습 구간 {len(lab):,}건, 양성 {int(lab.y.sum()):,} (패턴 외 {int(daily.unmatched.sum()):,}), 정상 {int((lab.y==0).sum()):,}")
print(f"정상:양성 전체 = 1:{(lab.y==0).sum()/lab.y.sum():.0f}")
daily[["pos", "unmatched", "neg", "neg_per_pos"]].describe().round(0)

## 1. 선택 (select) — 방식 5개 × 비율 3개 = 15개

`select/<이름>/selected.parquet` 가 만들어진다. 이미 있으면 건너뛰므로 셀 전체를 다시 실행해도 된다.
양성은 항상 전부 포함. 적용 위치(지도/구조)·가중치는 여기서 정하지 않는다 — train 에서 정한다.
시간: random·hard·strata 각 수 분, cssmc 는 군집 라벨 때문에 수십 분.

In [ ]:
# 공통 인자 — 학습 구간과 학습 구간용 피처(features_train)
common = ["--train", TRAIN, "--features", FEAT["train"], "--threads", str(THREADS), "--memory", MEMORY]

for r in ["10", "30", "100"]:
    select.main(["--method", "random", "--ratio", r] + common)                                              # RUS (기준선)
    select.main(["--method", "strata", "--ratio", r, "--axes", "day,amount,cross_bank"] + common)          # 다축 층화 3축
    select.main(["--method", "strata", "--ratio", r, "--axes", "day,amount,recent,cross_bank,new_pair"] + common)  # 5축
    select.main(["--method", "hard",   "--ratio", r, "--hard-frac", "0.5"] + common)                       # hard-negative 혼합
    select.main(["--method", "cssmc",  "--ratio", r, "--clusters", "30", "--candidates", "10"] + common)   # CSSMC

# 통계 참고용 (학습은 안 함): 샘플링 없음
# select.main(["--method", "none"] + common)


### 선택 결과 확인 — 일별 양성/정상 수, IPW

In [ ]:
name = f"random_r{R}"
sel = pd.read_parquet(L.SAMPLING / "select" / name / "selected.parquet")
print(json.dumps(json.loads((L.SAMPLING / "select" / name / "meta.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=1))
sel.groupby("day").agg(pos=("y", "sum"), neg=("y", lambda s: int((s == 0).sum())), ipw_neg=("ipw", lambda s: float(s[sel.loc[s.index, "y"] == 0].mean()))).head(10)

## 2. 감사 (audit) — 학습 없이 통계로 비교

피처는 `edge_features.parquet` 에서 DuckDB 로 읽는다 (memmap 캐시 불필요). 선택 하나당 parquet 스캔 2회 + 집계 → 수 분.

In [ ]:
audit.main(["--all", "--features", FEAT["train"], "--threads", str(THREADS), "--memory", MEMORY])

In [ ]:
import glob
A = pd.DataFrame({Path(p).parent.name: json.loads(open(p, encoding="utf-8").read()) for p in glob.glob(str(L.SAMPLING / "select" / "*" / "audit.json"))}).T
A["kl_raw"] = A.feature_raw.map(lambda d: d["kl_mean"]); A["kl_ipw"] = A.feature_ipw.map(lambda d: d["kl_mean"]); A["kl_max_feature"] = A.feature_raw.map(lambda d: d["kl_max_feature"])
A[["method", "ratio", "n_sel", "keep_rate", "kl_raw", "kl_ipw", "kl_max_feature", "daily_keep_ratio", "hour_jsd", "hard_frac_pop", "hard_frac_sel", "sibling_frac_pop", "sibling_frac_sel", "struct_pos_hop1_keep"]].round(5)

### 2-1. 통계 선별 — 같은 r 에서 random 과 비교

| 방식 | 목적 | 통과 기준 (아래 셀 상수로 조정) |
|---|---|---|
| random | 기준선 | — |
| strata, cssmc | 분포 보존 | `kl_raw` 가 random 의 `KL_RATIO` 배 이하 · `kl_ipw` < `KL_IPW_MAX` · 일별 보존 편차가 random 의 `DAILY_RATIO` 배 이하 · 예산 채움 ≥ `FILL_MIN` |
| hard | 경계 보존 (분포를 일부러 바꿈) | 통계로 판단 불가 → `hard_frac_sel` ≈ 0.5 면 학습 후보 |

통계는 "모집단을 얼마나 닮았나"까지만 알려준다. 세탁 구분에 도움이 되는 정상을 남겼는지는 학습으로 확인한다.

In [ ]:
KL_RATIO, KL_IPW_MAX, DAILY_RATIO, FILL_MIN = 0.7, 0.001, 2.0, 0.95

S = A[A.method != "none"].copy()
S["ratio"] = S.ratio.astype(float)
metas = {n: json.loads((L.SAMPLING / "select" / n / "meta.json").read_text(encoding="utf-8")) for n in S.index}
S["fill"] = [S.loc[n, "n_sel"] / (metas[n]["pos"] * metas[n]["ratio"]) for n in S.index]       # 선택 수 / 예산(양성 × r)
base = S[S.method == "random"].set_index("ratio")
S["kl_vs_random"]    = [S.loc[n, "kl_raw"] / base.loc[S.loc[n, "ratio"], "kl_raw"] for n in S.index]
S["daily_vs_random"] = [S.loc[n, "daily_keep_ratio"] / base.loc[S.loc[n, "ratio"], "daily_keep_ratio"] for n in S.index]

def verdict(x):
    if x.method == "random": return "기준선"
    if x.method == "hard":
        return "학습 후보 (경계 보존)" if abs(x.hard_frac_sel - 0.5) < 0.05 else f"확인: hard 비율 {x.hard_frac_sel:.2f}"
    bad = [why for ok, why in [(x.kl_vs_random <= KL_RATIO, "KL 개선 부족"), (x.kl_ipw < KL_IPW_MAX, "IPW 복원 안 됨"),
                               (x.daily_vs_random <= DAILY_RATIO, "날짜 편향"), (x.fill >= FILL_MIN, "예산 미달")] if not ok]
    return "통과" if not bad else "탈락: " + ", ".join(bad)

S["판정"] = S.apply(verdict, axis=1)
cols = ["method", "ratio", "n_sel", "fill", "kl_raw", "kl_vs_random", "kl_ipw", "kl_max_feature", "daily_keep_ratio", "daily_vs_random",
        "hour_jsd", "hard_frac_pop", "hard_frac_sel", "sibling_frac_sel", "struct_pos_hop1_keep", "판정"]
S.sort_values(["ratio", "method"])[cols].round(5)


### 피처 분포 그림 — 모집단 vs 선택 정상 (KL 큰 피처 위주)

In [ ]:
import matplotlib.pyplot as plt
name = f"random_r{R}"      # 다른 방식도 보려면 이름만 바꾼다 (예: f"cssmc_r{R}")
feat_dir = L.features_dir(FEAT["train"])
sel = pd.read_parquet(L.SAMPLING / "select" / name / "selected.parquet"); smp = sel[sel.y == 0]
pop_ids = np.sort(np.random.default_rng(0).choice(lab[lab.y == 0].edge_id.to_numpy(), 300_000, replace=False))
Xp, Xs = audit.read_features(con, feat_dir, pop_ids), audit.read_features(con, feat_dir, np.sort(smp.edge_id.to_numpy()[np.random.default_rng(0).choice(len(smp), min(len(smp), 300_000), replace=False)]))
fig, axes = plt.subplots(3, 6, figsize=(18, 8))
for j, ax in enumerate(axes.ravel()[:17]):
    lo, hi = np.quantile(Xp[:, j], [0.001, 0.999]); bins = np.linspace(lo, hi, 40)
    ax.hist(Xp[:, j], bins, density=True, alpha=.5, label="population"); ax.hist(Xs[:, j], bins, density=True, alpha=.5, label=name)
    ax.set_title(L.EDGE_FEATURES[j], fontsize=8); ax.tick_params(labelsize=6)
axes.ravel()[0].legend(fontsize=7); axes.ravel()[17].axis("off"); fig.tight_layout(); plt.show()

## 3. 학습 (train) — 통계 선별·LightGBM 비교 뒤 추린 조건만

학습 target 은 `features_train` 그래프, 검증 target 은 `features_val` 그래프로 만든다 (구간별 GraphStore 캐시가 따로 생김, 각 수십 분 1회).
캐시 크기는 그 구간 끝까지의 거래 수만큼 (학습 약 8GB, 검증 약 9GB, 테스트 약 10GB). 예전 `cache/features/`(전체 기간)는 지워도 된다.
`run.json` 있으면 건너뛴다 (다시 돌리려면 `runs/<run_name>/` 삭제). 아래 두 셀은 학습이 끝난 뒤 결과 확인용.

In [ ]:
# 완료된 실험 조건 전부 — run.json 요약 + 에폭별 검증 AP
import glob
runs = {}
for f in sorted(glob.glob(str(L.SAMPLING / "runs" / "*" / "run.json"))):
    r = json.loads(open(f, encoding="utf-8").read())
    runs[r["run_name"]] = dict(select=r["select"], apply=r["apply"], reweight=r["reweight"], pi_eff=r["pi_eff"], seed=r["seed"], pos_weight=round(r["pos_weight"], 2),
                               train_rows=r["train_rows"], epochs=r["epochs_run"], best_epoch=r["best_epoch"],
                               ms_per_target=round(r["train_sec_per_epoch"] / r["train_rows"] * 1000, 2), sec_per_epoch=round(r["train_sec_per_epoch"]),
                               gpu_gb=(round(r["gpu_peak_gb"], 1) if r["gpu_peak_gb"] else None), **{k: round(v, 4) for k, v in r["val"].items()})
summary = pd.DataFrame(runs).T
print(f"완료 {len(runs)}개 조건")
display(summary)

hist = pd.concat([pd.read_csv(Path(f).parent / "history.csv").assign(run=Path(f).parent.name) for f in glob.glob(str(L.SAMPLING / "runs" / "*" / "run.json"))], ignore_index=True) if runs else None
hist.pivot(index="epoch", columns="run", values="AP").round(4) if runs else None


### 검증 점수 분포 — 완료된 조건 전부 (양성/정상 점수 히스토그램, F1 임계값)

In [ ]:
# 검증 점수 분포 — 조건별 양성/정상 히스토그램과 F1 임계값
names = list(runs); cols = 3; rows_ = max(1, -(-len(names) // cols))
fig, axes = plt.subplots(rows_, cols, figsize=(6 * cols, 3.2 * rows_), squeeze=False); bins = np.linspace(0, 1, 50)
for ax, name in zip(axes.ravel(), names):
    vs = pd.read_parquet(L.SAMPLING / "runs" / name / "val_scores.parquet")
    ax.hist(vs[vs.y == 0].score, bins, weights=vs[vs.y == 0].w, alpha=.5, label="normal (weighted)", log=True)
    ax.hist(vs[vs.y == 1].score, bins, alpha=.6, label="laundering", log=True)
    ax.axvline(runs[name]["threshold"], color="k", ls=":"); ax.set_title(f"{name}\nAP {runs[name]['AP']:.4f}  thr {runs[name]['threshold']:.3f}", fontsize=8); ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(names):]: ax.axis("off")
axes[0, 0].legend(fontsize=7); fig.tight_layout(); plt.show()


### 학습 조건 — 주석을 풀며 실행

이전 1차 결과(runs/ 의 random r10·r30·r100)는 누수 있는 피처·옛 분할로 학습한 것이라 **다시 돌려야 한다** — `runs/`, `val/`, `results/` 폴더를 지우고 시작.
최종 후보는 마지막에 `--eval-test` 를 붙여 테스트 구간을 한 번만 평가한다.

In [ ]:
# 실험 조건 = (select 이름, apply, reweight, pi_eff, seed). run.json 있으면 건너뜀.
# 학습·검증 피처 폴더는 기본값(features_train / features_val)을 쓴다.
GRID = [
    # ── 비율 축 (RUS) ────────────────────────────────────────────────
    # ("random_r10",  "loss", "none", 0.25, 42),
    # ("random_r30",  "loss", "none", 0.25, 42),

    # ── 방식 축 (통계 선별 통과 + hard) ─────────────────────────────────
    # (f"strata_r{R}_day-amount-cross_bank",                 "loss", "none", 0.25, 42),
    # (f"strata_r{R}_day-amount-recent-cross_bank-new_pair", "loss", "none", 0.25, 42),
    # (f"hard_r{R}",                                         "loss", "none", 0.25, 42),
    # (f"cssmc_r{R}",                                        "loss", "none", 0.25, 42),

    # ── 양성 비중 축 (BEST 에서) ───────────────────────────────────────
    # (BEST, "loss", "none", 0.10, 42),
    # (BEST, "loss", "none", 0.50, 42),

    # ── 구조 샘플링 대조 (BEST 에서) ───────────────────────────────────
    # (BEST, "struct", "none", 0.25, 42),

    # ── 시드 반복 ────────────────────────────────────────────────────
    # (BEST, "loss", "none", 0.25, 43),
    # (BEST, "loss", "none", 0.25, 44),
]
EVAL_TEST = False     # 최종 후보를 돌릴 때만 True (테스트는 한 번만 본다)
for s, ap_, rw, pi, seed in GRID:
    train.main(["--select", s, "--apply", ap_, "--reweight", rw, "--pi-eff", str(pi), "--seed", str(seed),
                "--val", VAL, "--test", TEST, "--epochs", "6", "--patience", "2", "--batch", "32",
                "--threads", str(THREADS), "--memory", MEMORY] + (["--eval-test"] if EVAL_TEST else []))


## 4. 합치기 (collect) — 통계 감사 + 모델 지표 한 표

In [ ]:
df, summ = collect.main()
df[["run_name", "method", "ratio", "apply", "reweight", "pi_eff", "pos_weight", "train_rows", "AP", "F1", "recall_at_1pct", "unmatched_recall_at_1pct", "ECE",
    "kl_raw", "daily_keep_ratio", "hard_frac_sel", "train_sec_per_epoch", "gpu_peak_gb"]].sort_values(["method", "ratio", "apply", "reweight"]).round(4) if df is not None else None


## 판단 기준 (메모)

1. 통계 선별: strata·cssmc 는 같은 r 의 random 보다 `kl_raw` 가 확실히 작아야 학습할 가치가 있다. hard 는 통계로 거르지 않는다
2. 학습 비교는 순위 지표(AP, recall@1%, 패턴 외 recall@1%)로. ECE·threshold 는 정상을 줄인 만큼 확률이 부풀려져 조건 간 비교 불가
3. 같은 r·pi_eff 에서 방식 간 AP 차이가 에폭 간 흔들림(±0.05)보다 작으면 "차이 없음" — 시드 반복으로 확인
4. 테스트(`--eval-test`)는 최종 후보에만, 한 번만